# E-Commerce Customer Analytics & Machine Learning Pipeline
An end-to-end data pipeline covering:
1. **Data Cleaning & Preprocessing** (Handling nulls, duplicates, cancellations, invalid transactions)
2. **Feature Engineering & RFM Modeling** (Recency, Frequency, Monetary value metrics)
3. **Data Warehousing & Star Schema Design** (Fact & Dimension tables with surrogate keys)
4. **OLAP Operations** (Roll-up, Drill-down, Slice, Dice, Pivot)
5. **Customer Churn Modeling** (Lookback/observation window, Random Forest classification & evaluation)


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print("Libraries imported successfully!")

In [ ]:
file_path = '/kaggle/input/datasets/bhagirathchaudhary/e-commerce-data/data.csv'
if not os.path.exists(file_path):
    matching = glob.glob('/kaggle/input/**/data.csv', recursive=True)
    if matching:
        file_path = matching[0]
    else:
        file_path = 'data.csv'

df = pd.read_csv(file_path, encoding='ISO-8859-1')

print(f"Dataset loaded successfully! Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

In [ ]:
print("--- Column Summary & Null Counts ---")
print(df.isnull().sum())
print("\n--- Statistical Summary ---")
display(df.describe(include='all'))

In [ ]:
df_clean = df.copy()

df_clean.columns = df_clean.columns.str.strip()

before_dups = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before_dups - len(df_clean):,} duplicate rows. Remaining: {len(df_clean):,}")

df_clean = df_clean.dropna(subset=['StockCode'])
df_clean['Description'] = df_clean['Description'].fillna('Unknown Product')
df_clean['Country'] = df_clean['Country'].fillna('Unknown Country')

df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str).str.strip()
df_clean['CustomerID'] = pd.to_numeric(df_clean['CustomerID'], errors='coerce').astype('Int64')
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['UnitPrice'] = pd.to_numeric(df_clean['UnitPrice'], errors='coerce')
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')

df_clean = df_clean.dropna(subset=['InvoiceDate'])

df_clean['IsCancelled'] = df_clean['InvoiceNo'].str.startswith('C')

df_sales = df_clean[
    (~df_clean['IsCancelled']) &
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
].copy()

q1 = df_sales['UnitPrice'].quantile(0.25)
q3 = df_sales['UnitPrice'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
df_sales['PriceOutlier'] = df_sales['UnitPrice'] > upper_bound

print(f"Total sales records cleaned: {len(df_sales):,} (out of {len(df):,} original)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=axes[0])
axes[0].set_title("Missing Values (Original Dataset)")

sns.boxplot(x=df_sales['UnitPrice'], ax=axes[1], color='#3498db')
axes[1].set_title("Boxplot of Unit Price (df_sales)")
axes[1].set_xlabel("Unit Price")

plt.tight_layout()
plt.show()

In [ ]:
df_sales['TotalAmount'] = df_sales['Quantity'] * df_sales['UnitPrice']

df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month'] = df_sales['InvoiceDate'].dt.month
df_sales['MonthName'] = df_sales['InvoiceDate'].dt.month_name()
df_sales['Day'] = df_sales['InvoiceDate'].dt.day
df_sales['DayName'] = df_sales['InvoiceDate'].dt.day_name()
df_sales['Hour'] = df_sales['InvoiceDate'].dt.hour
df_sales['DateKey'] = df_sales['InvoiceDate'].dt.strftime('%Y%m%d').astype(int)

reference_date = df_sales['InvoiceDate'].max()

customer_metrics = df_sales.dropna(subset=['CustomerID']).groupby('CustomerID').agg(
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    TotalRevenue=('TotalAmount', 'sum'),
    AverageOrderValue=('TotalAmount', 'mean'),
    LastPurchaseDate=('InvoiceDate', 'max')
).reset_index()

customer_metrics['PurchaseFrequency'] = customer_metrics['TotalOrders']
customer_metrics['AverageQuantityPerOrder'] = customer_metrics['TotalQuantity'] / customer_metrics['TotalOrders']
customer_metrics['Recency'] = (reference_date - customer_metrics['LastPurchaseDate']).dt.days

customer_metrics['RecencyCategory'] = pd.qcut(customer_metrics['Recency'].rank(method='first'), q=3, labels=['Recent', 'Moderate', 'At Risk'])
customer_metrics['FrequencyCategory'] = pd.qcut(customer_metrics['TotalOrders'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])
customer_metrics['MonetaryCategory'] = pd.qcut(customer_metrics['TotalRevenue'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])

print("Customer RFM metrics engineered:")
display(customer_metrics.head())

In [ ]:
dim_customer = df_sales[['CustomerID', 'Country']].dropna(subset=['CustomerID']).drop_duplicates(subset=['CustomerID'])
dim_customer = dim_customer.merge(customer_metrics, on='CustomerID', how='left').reset_index(drop=True)
dim_customer['CustomerKey'] = range(1, len(dim_customer) + 1)

product_metrics = df_sales.groupby('StockCode').agg(
    TotalUnitsSold=('Quantity', 'sum'),
    TotalRevenue=('TotalAmount', 'sum'),
    TotalTransactions=('InvoiceNo', 'nunique'),
    AverageSellingPrice=('UnitPrice', 'mean')
).reset_index()

dim_product = df_sales[['StockCode', 'Description', 'UnitPrice']].drop_duplicates(subset=['StockCode']).reset_index(drop=True)
dim_product = dim_product.merge(product_metrics, on='StockCode', how='left')
dim_product['ProductKey'] = range(1, len(dim_product) + 1)

dim_date = df_sales[['DateKey', 'InvoiceDate', 'Year', 'Month', 'MonthName', 'Day', 'DayName', 'Hour']].drop_duplicates(subset=['DateKey']).reset_index(drop=True)

customer_map = dim_customer.set_index('CustomerID')['CustomerKey']
product_map = dim_product.set_index('StockCode')['ProductKey']

fact_sales = df_sales[[
    'InvoiceNo', 'CustomerID', 'StockCode', 'DateKey', 'Quantity', 'UnitPrice', 'TotalAmount'
]].copy()

fact_sales['CustomerKey'] = fact_sales['CustomerID'].map(customer_map)
fact_sales['ProductKey'] = fact_sales['StockCode'].map(product_map)

fact_sales_clean = fact_sales.dropna(subset=['CustomerKey', 'ProductKey', 'DateKey']).copy()
fact_sales_clean['CustomerKey'] = fact_sales_clean['CustomerKey'].astype(int)
fact_sales_clean['ProductKey'] = fact_sales_clean['ProductKey'].astype(int)
fact_sales_clean['DateKey'] = fact_sales_clean['DateKey'].astype(int)

fact_sales_final = fact_sales_clean[['InvoiceNo', 'CustomerKey', 'ProductKey', 'DateKey', 'Quantity', 'UnitPrice', 'TotalAmount']]

print(f"Fact Table: {fact_sales_final.shape}")
print(f"Customer Dimension: {dim_customer.shape}")
print(f"Product Dimension: {dim_product.shape}")
print(f"Date Dimension: {dim_date.shape}")

In [ ]:
os.makedirs('/kaggle/working', exist_ok=True)
fact_sales_final.to_csv('/kaggle/working/fact_sales.csv', index=False)
dim_customer.to_csv('/kaggle/working/dim_customer.csv', index=False)
dim_product.to_csv('/kaggle/working/dim_product.csv', index=False)
dim_date.to_csv('/kaggle/working/dim_date.csv', index=False)
print("Warehouse CSV files exported successfully.")

In [ ]:
merged_olap = fact_sales_final.merge(dim_date[['DateKey', 'Year', 'Month', 'MonthName']], on='DateKey', how='left')

olap_rollup = merged_olap.groupby('Year').agg(
    TotalRevenue=('TotalAmount', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    TotalTransactions=('InvoiceNo', 'nunique')
).reset_index()

olap_drilldown = merged_olap.groupby(['Year', 'Month', 'MonthName']).agg(
    TotalRevenue=('TotalAmount', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    TotalTransactions=('InvoiceNo', 'nunique')
).reset_index().sort_values(['Year', 'Month'])

target_year = 2011
olap_slice = olap_drilldown[olap_drilldown['Year'] == target_year].copy()

olap_dice = merged_olap[
    (merged_olap['Year'] == 2011) &
    (merged_olap['Month'].isin([1, 2, 3])) &
    (merged_olap['TotalAmount'] > 100)
].groupby(['Year', 'Month']).agg(
    TotalRevenue=('TotalAmount', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    TotalTransactions=('InvoiceNo', 'nunique')
).reset_index()

month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
olap_pivot = merged_olap.pivot_table(index='Year', columns='MonthName', values='TotalAmount', aggfunc='sum', fill_value=0)
olap_pivot = olap_pivot.reindex(columns=month_order, fill_value=0)

print("OLAP Roll-up Summary:")
display(olap_rollup)
print("\nOLAP Pivot Matrix:")
display(olap_pivot)

In [ ]:
max_date = df_sales['InvoiceDate'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

df_hist = df_sales[(df_sales['InvoiceDate'] < cutoff_date) & (df_sales['CustomerID'].notnull())].copy()

df_future = df_sales[(df_sales['InvoiceDate'] >= cutoff_date) & (df_sales['InvoiceDate'] <= max_date) & (df_sales['CustomerID'].notnull())].copy()

churn_df = df_hist.groupby('CustomerID').agg(
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum'),
    TotalRevenue=('TotalAmount', 'sum'),
    AverageOrderValue=('TotalAmount', 'mean'),
    AverageUnitPrice=('UnitPrice', 'mean'),
    LastPurchaseDate=('InvoiceDate', 'max')
).reset_index()

churn_df['Recency'] = (cutoff_date - churn_df['LastPurchaseDate']).dt.days

future_buyers = set(df_future['CustomerID'].unique())
churn_df['Churn'] = churn_df['CustomerID'].apply(lambda cid: 0 if cid in future_buyers else 1)

print(f"Customer Churn Dataset: {len(churn_df)} customers")
print(f"Class Distribution:\n{churn_df['Churn'].value_counts(normalize=True).round(3)}")

In [ ]:
feature_cols = ['TotalOrders', 'TotalQuantity', 'TotalRevenue', 'AverageOrderValue', 'AverageUnitPrice', 'Recency']
X = churn_df[feature_cols]
y = churn_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

rf_clf.fit(X_train, y_train)

y_pred = rf_clf.predict(X_test)
y_prob = rf_clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

print("="*45)
print("       CHURN MODEL EVALUATION METRICS        ")
print("="*45)
print(f"Accuracy   : {acc * 100:.2f}%")
print(f"Precision  : {prec * 100:.2f}%")
print(f"Recall     : {rec * 100:.2f}%")
print(f"F1-Score   : {f1 * 100:.2f}%")
print(f"ROC-AUC    : {roc:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Retained (0)', 'Churned (1)'],
            yticklabels=['Retained (0)', 'Churned (1)'])
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

feat_imp = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=True)
feat_imp.plot(kind='barh', color='#2ecc71', ax=axes[1])
axes[1].set_title("Random Forest Feature Importances")
axes[1].set_xlabel("Relative Importance Score")

plt.tight_layout()
plt.show()